# 09 - Ensemble seletivo e gate final

Combina somente modelos ja executados, escolhe a politica na calibracao e mede uma vez nos cinco dias internos posteriores. O holdout externo permanece fechado.

In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
ROOT=Path.cwd(); NB_DIR=ROOT/'notebooks' if (ROOT/'notebooks').is_dir() else ROOT; sys.path.insert(0,str(NB_DIR))
from produtividade_30d import I, carregar_fontes, preparar_dataset, dividir_por_dia, buscar_limiares, predicao_seletiva, metricas
OUT=Path(os.environ.get('KV_30D_OUTPUT_DIR',NB_DIR/'outputs'/'productivity_30d'))
leve=pd.read_csv(OUT/'modelos_leves_predicoes.csv'); lora_path=OUT/'lora_predicoes.csv'
dados=leve.copy()
if lora_path.is_file(): dados=dados.merge(pd.read_csv(lora_path),on=['id','dia','split','y_true','y_baseline'],how='inner')
dados['prob_I_catalogo']=np.where(dados.y_baseline==I,0.85,0.15)
candidatos={
 'catalogo':{'prob_I_catalogo':1.0},
 'leve_combinado':{'prob_I_combinado':1.0},
 'catalogo+estruturado':{'prob_I_catalogo':0.6,'prob_I_estruturado':0.4},
 'catalogo+texto+estruturado':{'prob_I_catalogo':0.5,'prob_I_texto':0.25,'prob_I_estruturado':0.25},
}
if 'prob_I_lora' in dados: candidatos['catalogo+combinado+LoRA']={'prob_I_catalogo':0.45,'prob_I_combinado':0.25,'prob_I_lora':0.30}
cal=dados[dados.split=='calibracao'].copy(); teste=dados[dados.split=='teste_interno'].copy(); resultados=[]; escolhas=[]
for nome,pesos in candidatos.items():
    prob_cal=sum(cal[c]*w for c,w in pesos.items()); escolha=buscar_limiares(cal,prob_cal.to_numpy())
    prob_teste=sum(teste[c]*w for c,w in pesos.items())
    for parte,df,prob in [('calibracao',cal,prob_cal),('teste_interno',teste,prob_teste)]:
        pred=predicao_seletiva(prob.to_numpy(),escolha['limiar_I'],escolha['limiar_P'])
        resultados.append({'ensemble':nome,'parte':parte,'limiar_I':escolha['limiar_I'],'limiar_P':escolha['limiar_P'],'passou_calibracao':escolha['passou'],**metricas(df,pred)})
    escolhas.append((bool(escolha['passou']),escolha['metricas']['coverage'],escolha['metricas']['f1_I'],nome))
resultado=pd.DataFrame(resultados)
aprovados=[item for item in escolhas if item[0]]; vencedor=max(aprovados)[-1] if aprovados else None
resultado['selecionado_calibracao']=resultado.ensemble.eq(vencedor) if vencedor else False
resultado.to_csv(OUT/'ensemble_metricas.csv',index=False)
decisao={'selecionado':vencedor,'criterio':'passa todos os gates; depois maior coverage e F1_I','aprovado_para_producao':False,'motivo':('nenhuma candidata passou a calibracao' if vencedor is None else 'teste interno usa proxy derivada; falta holdout binario novo e independente')}
(OUT/'ensemble_decisao.json').write_text(json.dumps(decisao,indent=2,ensure_ascii=False),encoding='utf-8')
display(resultado); print(json.dumps(decisao,indent=2,ensure_ascii=False))